# **Start**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title  { display-mode: "form" }
#@title  { display-mode: "form" }
#@title  { vertical-output: true, display-mode: "form" }
#@markdown Select The Best

use_pearson_corr = "False" #@param ["True", "False"]
Data_to_use = "6_bands" #@param ["6_bands", "65_bands", "372_bands", "Planet_hisar", "Pavia", "Indian_Pines", "Hisar_sentinel"]
use_pearson_corr = True if use_pearson_corr in [True, "True"] else False
P_S = "9"  #@param [3, 9, 15, 21]

P_S = int(P_S)
Targeted_accuracy = "0.985" #@param [0.97, 0.98, 0.985, 0.99, 0.995]
Min_trainable_epoch = 20 #@param [20, 25, 30, 35, 40, 50, 100]

# --- SACP Parameters ---
SACP_ALPHA = 0.05   #@param {type:"number"} # Error rate (0.05 = 95% Coverage)
SACP_LAMBDA = 0.5   #@param {type:"number"} # Smoothing weight (0 to 1)
SACP_K = 1          #@param {type:"integer"} # Number of smoothing iterations

train_percent = 75                  #@param [75, 80, 90]
epoch = 100                         #@param [100, 200, 300, 400, 500]

LR_START = 3e-3
LR_MAX = 6e-3
LR_MIN = 1e-5

batch_size = 128
BATCH_SIZE = 128
dropout_rate = "0.25"                 #@param [0.1, 0.2, 0.25, 0.3, 0.4, 0.5]

P_S = int(P_S)
epoch = int(epoch)
Min_trainable_epoch = int(Min_trainable_epoch)
Targeted_accuracy = float(Targeted_accuracy)
train_percent = int(train_percent)
dropout_rate = float(dropout_rate)
shifts = int(1/dropout_rate)
assert epoch >= Min_trainable_epoch/dropout_rate, f"For Min_trainable_epoch: {Min_trainable_epoch} & dropout_rate: {dropout_rate}, minimum number of `epoch` should be {Min_trainable_epoch/dropout_rate}"

In [ ]:
!pip install spectral
!pip install tensorflow_addons

In [ ]:
!pip install tensorflow

In [ ]:
import pandas as pd           # for csv files and dataframes
import numpy as np            # Linear Algebra tools
import matplotlib.pyplot as plt  # for ploting graphs and curve
from matplotlib import colors, cm, gridspec
import scipy.stats as st
import scipy.io as si         # for inputing matlab files
from random import shuffle    # for shuffling dataset
import seaborn as sns
from tqdm import tqdm         # Progress bar for SACP
sns.set()

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix   #Confusion matrix creation
from sklearn.metrics import accuracy_score, cohen_kappa_score
from sklearn.metrics import classification_report

import datetime
import time
import warnings
import math
import os
import gc
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import openpyxl
from openpyxl import Workbook, load_workbook
from openpyxl.drawing.image import Image as XLImage
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.python.util.tf_export import keras_export
from tensorflow.keras import Sequential, layers
from tensorflow.keras.layers import Input, Add, Multiply, Reshape, Dense, Activation, BatchNormalization, Flatten, Dropout, concatenate, Lambda
from tensorflow.keras.layers import Conv2D, AveragePooling2D, MaxPooling2D,  GlobalAveragePooling2D, GlobalAvgPool2D, DepthwiseConv2D, SeparableConv2D, MaxPool2D, UpSampling2D
from tensorflow.keras.layers import Conv2DTranspose, add, multiply
from tensorflow.python.ops import array_ops
from tensorflow.python.keras.utils import control_flow_util
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.models import load_model, Model
from tensorflow.keras import optimizers
from tensorflow.keras.initializers import glorot_uniform, Ones
from tensorflow.keras.models import Model
from tensorflow.keras.utils import plot_model
from keras.regularizers import l2

import spectral

np.random.seed(1337)          # to get reproducible results
base_path = "/content/drive/MyDrive/Uncertainty/CNN Models/6 bands/SingleHead//"

try:
    os.mkdir(base_path)
except:
    pass

try:
    os.mkdir(base_path + "With Pearson correlation/") if use_pearson_corr else os.mkdir(base_path + "Without Pearson correlation/")
except:
    pass

if use_pearson_corr:
    folder_path = base_path + "With Pearson correlation/" + str(Data_to_use) + "/"
    try:
        os.mkdir(folder_path)
    except:
        pass
else:
    folder_path = base_path + "Without Pearson correlation/" + str(Data_to_use) + "/"
    try:
        os.mkdir(folder_path)
    except:
        pass

In [ ]:
try:
    os.mkdir(folder_path + "Trained models")
    os.mkdir(folder_path + "Results")
except:
    pass

In [ ]:
Normalize_data = True
if Data_to_use == "6_bands":
    H, W, B = 330, 307, 6
    x = np.array(pd.read_csv('/content/drive/My Drive/m_p/data/multispectral/data.csv'))
    y = np.array(pd.read_csv('/content/drive/My Drive/m_p/data/multispectral/ref.csv'))

if Data_to_use == "65_bands":
    H, W, B = 512, 512, 65
    x = si.loadmat('/content/drive/My Drive/m_p/data/Dias/DIAS.mat')['DIAS']
    y = si.loadmat('/content/drive/My Drive/m_p/data/Dias/DIAS_ref.mat')['DIAS_ref']

if Data_to_use == "372_bands":
    H, W, B = 1101, 566, 372
    x = si.loadmat('/content/drive/My Drive/m_p/data/372 band/full.mat')['full']
    y = si.loadmat('/content/drive/My Drive/m_p/data/372 band/full_gt.mat')['full_gt']
    x[x < 0.0] = 0.0
    x[x > 1.0] = 1.0
    Normalize_data = False
    #x = x[:, selective_bands]

if Data_to_use == "Planet_hisar":
    H, W, B = 1733, 2647, 4
    x = pd.read_csv('/content/drive/My Drive/m_p/data/planet_data_hisar/planet.csv', header=None)
    y = pd.read_csv('/content/drive/My Drive/m_p/data/planet_data_hisar/planetgt.csv', header=None)

if Data_to_use == "Pavia":
    H, W, B = 610, 340, 103
    x = si.loadmat('/content/drive/MyDrive/m_p/data/pavia/paviauni.mat')['paviaU']
    y = si.loadmat('/content/drive/MyDrive/m_p/data/pavia/paviauni_gt.mat')['paviaU_gt']

if Data_to_use == "Indian_Pines":
    H, W, B = 145, 145, 220
    x = si.loadmat('/content/drive/My Drive/m_p/data/Indian_pines/Indian_pines.mat')['indian_pines']
    y = si.loadmat('/content/drive/My Drive/m_p/data/Indian_pines/Indian_pines_gt.mat')['indian_pines_gt']

if Data_to_use == "Hisar_sentinel":
    H, W, B = 722, 1014, 4
    x = np.array(si.loadmat('/content/drive/My Drive/m_p/data/Hissar_25%/datas.mat')['datas'])
    y = np.array(pd.read_csv('/content/drive/My Drive/m_p/data/Hissar_25%/fullgt.csv'))

y_shape = np.array(y).shape[0]
x, y = np.array(x).reshape(H,W,B), np.array(y).reshape(H,W)
x = x.astype('float16')
#########
print("minimum value in raw data is :", x.min())
print("maximum value in raw data is :", x.max(), "\n")
print("shape of raw data: ", x.shape)
print("shape of target data: ", y.shape, "\n")

# Normalizing the data between 0,1
if Normalize_data:
    for i in range(B):
        band_min = x[:,:,i].min()
        band_max = x[:,:,i].max()
        band_range = band_max - band_min
        x[:,:,i] = (x[:,:,i] - band_min)/band_range

# **Data**

In [ ]:
# --- MODIFIED DATA GENERATION FOR SACP ---
pad_width = int((P_S-1)/2)
padded_x = np.pad(x,[(pad_width,pad_width),(pad_width,pad_width),(0,0)],'edge')

X, Y, INDICES = [], [], [] # Added INDICES list to track spatial coordinates

for a in range(H):
  for b in range(W):
    if(y[a][b]!= 0):
      patch = padded_x[a:a+P_S,b:b+P_S,:]
      X.append(patch)
      Y.append(y[a][b]-1)
      INDICES.append((a, b)) # Store coordinates (row, col)

X = np.array(X)
Y = np.array(Y)
INDICES = np.array(INDICES)

num_classes = len(np.unique(y))-1
Approximate_rgb_img = x[:,:,[B//2-1, B//2, B//2+1]]*255

print()
print("minimum value in x is :", x.min())
print("maximum value in x is :", x.max())
print()
print("shape of x: ", x.shape)
print("shape of y: ", y.shape)
print()
print("shape of padded_x: ", padded_x.shape, "\n")
print(f"{len(Y)*100/y_shape:.4f}% of data is labeled with {num_classes} classes")

# --- SACP DATA SPLIT ---
# First split: Train vs Remainder
x_train, x_remain, y_train, y_remain, idx_train, idx_remain = train_test_split(
    X, Y, INDICES, train_size=(train_percent/100), stratify=Y, random_state=10
)

# Second split: Remainder -> Calibration (50%) + Test (50%)
# We need a separate calibration set for Conformal Prediction
x_calib, x_test, y_calib, y_test, idx_calib, idx_test = train_test_split(
    x_remain, y_remain, idx_remain, train_size=0.5, stratify=y_remain, random_state=10
)

print("-"*30)
print(f"Train shapes: {x_train.shape}")
print(f"Calib shapes: {x_calib.shape} (For SACP Calibration)")
print(f"Test shapes:  {x_test.shape} (For Evaluation)")
print("-"*30)

In [ ]:
class_labels, value_counts = np.unique(y.reshape(-1,1), return_counts = True)
plt.figure(figsize = (15,5))
plt.bar(class_labels[1:]-1, value_counts[1:])
plt.xticks(class_labels, rotation = 0)
plt.xlabel("classes")
plt.ylabel("Number of pixels")
plt.title("Distribution Plot")
plt.show()

In [ ]:
gc.collect()

# **Functions**

In [ ]:
def predict_half_image(model, padded_x, H, W_range, B, P_S, num_classes):
    y_hat = np.zeros((H,W_range))
    y_prob = np.zeros((H, W_range, num_classes))
    for j in range(W_range):
        patchs = np.zeros((H, P_S, P_S, B))
        for i in range(H):
            patchs[i,:,:,:] = padded_x[i:i+P_S, j:j+P_S, :]
        prob = model.predict(patchs, verbose = -1)
        # y_pred = np.argmax(prob,axis=1)+1
        y_prob[:,j,:] = prob
        y_hat[:,j] = np.argmax(prob,axis=1)+1
        del patchs, prob
        gc.collect()
    return y_hat, y_prob


def predict_image(model, padded_x, H, W, B, P_S, num_classes):
    start_time = time.time()
    half_1 = padded_x.shape[1]//2 + pad_width+1
    half_2 = padded_x.shape[1]//2 - pad_width-1
    padded_x_1st_half = padded_x[:,:half_1, :]
    padded_x_2nd_half = padded_x[:,half_2:, :]
    del padded_x
    gc.collect()

    y_hat = np.zeros((H,W))
    y_prob = np.zeros((H,W, num_classes))
    y_hat[:,:W//2], y_prob[:,:W//2,:] = predict_half_image(model, padded_x_1st_half, H, W//2, B, P_S, num_classes)
    # del padded_x_1st_half
    print("image predicted ███████████████████████ 50%")
    gc.collect()
    width = W//2 if W//2 == W/2 else W//2+1
    y_hat[:,W//2:], y_prob[:,W//2:,:] = predict_half_image(model, padded_x_2nd_half, H, width, B, P_S, num_classes)
    # del padded_x_2nd_half
    print("image predicted ██████████████████████████████████████████████ 100%")
    gc.collect()
    end_time = time.time()
    time_taken = end_time - start_time
    time_min = time_taken//60
    time_sec = time_taken - time_min*60
    print(f'total time taken is: {time_min} min {time_sec:.2f} sec.')
    y_hat = np.reshape(y_hat, (H,W))
    y_prob = np.reshape(y_prob, (H, W, num_classes))
    print("shape of predicted image is: ", y_hat.shape)
    return y_hat, y_prob

In [ ]:
def predict_half_image_prob(model, padded_x, H, W_range, B, P_S):
    y_prob = np.zeros((H, W_range, num_classes))
    for j in range(W_range):
        patchs = np.zeros((H, P_S, P_S, B))
        for i in range(H):
            patchs[i,:,:,:] = padded_x[i:i+P_S, j:j+P_S, :]
        y_pred_prob = model.predict(patchs, verbose = -1)
        y_prob[:,j,:] = y_pred_prob
        del patchs, y_pred_prob
        gc.collect()
    return y_prob

def probabilistic_outputs(model, padded_x, H, W, B, P_S):
    start_time = time.time()
    half_1 = padded_x.shape[1]//2 + pad_width+1
    half_2 = padded_x.shape[1]//2 - pad_width-1
    padded_x_1st_half = padded_x[:,:half_1, :]
    padded_x_2nd_half = padded_x[:,half_2:, :]
    del padded_x
    gc.collect()

    y_hat = np.zeros((H,W, num_classes))
    y_hat[:,:W//2,:] = predict_half_image_prob(model, padded_x_1st_half, H, W//2, B, P_S)
    print("image predicted ███████████████████████ 50%")
    gc.collect()
    width = W//2 if W//2 == W/2 else W//2+1
    y_hat[:,W//2:,:] = predict_half_image_prob(model, padded_x_2nd_half, H, width, B, P_S)
    print("image predicted ██████████████████████████████████████████████ 100%")
    gc.collect()
    end_time = time.time()
    time_taken = end_time - start_time
    time_min = time_taken//60
    time_sec = time_taken - time_min*60
    print(f'total time taken is: {time_min} min {time_sec:.2f} sec.')
    print("shape of predicted image is: ", y_hat.shape)
    return y_hat

In [ ]:
def predict(model, x_test):
    y_pred = np.argmax(model.predict(x_test, verbose = -1), axis = -1)
    return y_pred

In [ ]:
def plot_accuracy_loss_curve(history, use_pearson_corr = None, folder_path = None):
    train_loss = history.history['loss']
    val_loss = history.history['val_loss']
    train_accuracy = history.history['accuracy']
    val_accuracy = history.history['val_accuracy']

    fig = plt.figure(figsize = (24,8))
    ax = plt.subplot(1,1,1)
    ax2 = ax.twinx()
    ax.plot(train_accuracy, color='blue', marker='o', linewidth=1.5, markersize = 2,  label = 'train_accuracy')
    ax.plot(val_accuracy, color='green', marker='o', linewidth=1.5, markersize = 2, label = 'val_accuracy')
    ax.grid()
    plt.xlabel('no. of epoches')
    plt.ylabel('accuracy')
    ax.legend()
    ax2.plot(train_loss, color = 'black', marker='o', linewidth=1.5, markersize = 2, label = 'train_loss')
    ax2.plot(val_loss, color = 'red', marker='o', linewidth=1.5, markersize = 2, label = 'val_loss')
    ax2.grid()
    plt.xlabel('no. of epoches')
    plt.ylabel('loss')
    plt.title('accuracy and loss plot for model performance')
    ax2.legend()
    plt.show()
    if folder_path:
        results_dir = os.path.join(folder_path, "Results")
        os.makedirs(results_dir, exist_ok=True) # Create the directory if it doesn't exist
        if use_pearson_corr:
            path = os.path.join(results_dir, "Pearson_Corr " + str(train_percent) + "% ps_"+ str(P_S) + " accuracy_loss.png")
        else:
            path = os.path.join(results_dir, str(train_percent) + "% ps_"+ str(P_S) + " accuracy_loss.png")
        fig.savefig(path)

In [ ]:
def performance_meausures(y_test, y_pred, tt, *parameters_summary, folder_path = None):
    Total_params, Trainable_params, Non_trainable_params = parameters_summary
    accuracy = accuracy_score(y_test, y_pred)
    kappa=cohen_kappa_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred).astype('int32')
    cr = classification_report(y_test, y_pred, output_dict=True)
    df_cr = pd.DataFrame(cr).T
    df_score = pd.DataFrame({'accuracy score: ' : [accuracy], 'Cohen_Kappa score: ' : [kappa], "Training Time: " : [tt]}).T
    df_summary = pd.DataFrame({'Total_params: ': [Total_params], 'Trainable_params: ' : [Trainable_params], 'Non_trainable_params: ':[Non_trainable_params]}).T

    spec = gridspec.GridSpec(ncols = 2, nrows = 2, width_ratios=[1,3], wspace = 0.5, hspace = 0.5, height_ratios=[7,1])

    fig = plt.figure(figsize = (24,10))

    ax1 = fig.add_subplot(spec[0])
    ax1.set_title('classification report')
    sns.heatmap(df_cr, cmap = 'Blues', cbar = False, annot = True, fmt=' .5g', ax = ax1)

    ax2 = fig.add_subplot(spec[1])
    ax2.set_title('confusion matrix')
    ax2.set_xlabel('predicted class')
    ax2.set_ylabel('actual class')
    sns.heatmap(cm, cmap = 'Blues', cbar = False, annot = True, fmt=' .5g', ax = ax2)

    ax3 = fig.add_subplot(spec[2])
    sns.heatmap(df_score, cmap = 'Blues', cbar = False, annot = True, fmt=' .5g', ax = ax3)
    ax3.set_xticks([])

    ax4 = fig.add_subplot(spec[3])
    sns.heatmap(df_summary, cmap = "Blues", cbar = False, annot = True, fmt=' .10g', ax = ax4)
    ax4.set_xticks([])

    if folder_path:
        path = folder_path + "Results/" + str(train_percent) + "% ps_" + str(P_S) +" Performance Measure.png"
        fig.savefig(path)

In [ ]:
def measure_uncertainty(model, x_test, y_test):
    y_prob = model.predict(x_test, verbose = -1)

    """Sorting is as follow
    y_prob = [[11 12 13 22]
              [31 10 33  7]
              [21  7 23 14]]

    temp_prob will sort the y_prob along rows
    temp_prob = [[11 12 13 22]
                 [7  10 31 33]
                 [7  14 21 23]]

    Now sort the y_prob based on the last column of temp_pred i.e. last col is [22, 33, 23] and sorting will be [22, 23, 33] ---> [0, 2, 1]
     y_prob = [[11 12 13 22]
               [21  7 23 14]
               [31 10 33  7]]
    """
    temp_pred = np.sort(y_prob, axis = -1)      # sort the probabilities along rows. Higher prob values are along last column
    y_prob = y_prob[temp_pred[:,-1].argsort()]  # Arange the rows of predicted prob in assending position

    num = y_prob.shape[0]
    # observed = y_prob.max(axis = -1)
    observed = np.sort(y_prob, axis = -1)[:,-1]    # class predicted according to 1st maximum prob
    observed_1 = np.sort(y_prob, axis = -1)[:,-2]  # class predicted according to 2nd maximum prob
    observed_2 = np.sort(y_prob, axis = -1)[:,-3]  # class predicted according to 3rd maximum prob

    mean = y_prob.mean(axis = -1)
    std = y_prob.std(axis = -1)
    plt.figure(figsize = (24,8))
    plt.plot(np.arange(1, num+1, 1), observed - observed_1, color = "black", label = "certaintity")
    plt.plot(np.arange(1, num+1, 1), observed, color = "blue", label = "observed")
    plt.plot(np.arange(1, num+1, 1), observed_1, color = "purple", label = "observed_1")
    plt.plot(np.arange(1, num+1, 1), observed_2, color = "cyan", label = "observed_2")
    plt.plot(np.arange(1, num+1, 1), mean, color = "red", label = "mean")
    plt.plot(np.arange(1, num+1, 1), mean + 2*std, color = "green", label = "mean + 2*std")
    plt.plot(np.arange(1, num+1, 1), mean - 2*std, color = "green", label = "mean - 2*std")
    plt.legend()
    plt.title("Predictions based on normal model")
    plt.show()

In [ ]:
class Pearson_correlation_masked(layers.Layer):
    def __init__(self, P_S = 9, **kwargs):
        super(Pearson_correlation_masked, self).__init__(**kwargs)
        self.P_S = P_S

    def call(self, inputs):
        self.loc = (self.P_S)//2
        self.inputs = inputs
        self.channels = self.inputs.shape[-1]
        self.x_mean = tf.repeat(tf.math.reduce_mean(self.inputs, axis = -1, keepdims=True), repeats = self.channels, axis = -1)

        self.y = tf.repeat(tf.repeat(self.inputs[:,self.loc:self.loc+1, self.loc:self.loc+1, :], repeats = self.P_S, axis = -2), repeats = self.P_S, axis = -3)

        self.y_mean = tf.repeat(tf.math.reduce_mean(self.y, axis = -1, keepdims = True), repeats = self.channels, axis = -1)

        self.a = tf.math.subtract(self.inputs, self.x_mean)
        self.b = tf.math.subtract(self.y, self.y_mean)
        self.ab = tf.math.multiply(self.a,self.b)
        self.num = tf.math.reduce_sum(self.ab, axis = -1, keepdims = True)

        self.a_new = tf.math.reduce_sum(tf.math.multiply(self.a, self.a), axis = -1, keepdims = True)
        self.b_new = tf.math.reduce_sum(tf.math.multiply(self.b, self.b), axis = -1, keepdims = True)
        self.deno = tf.math.sqrt(tf.math.multiply(self.a_new, self.b_new))

        self.corr = tf.math.divide(self.num, self.deno)

        self.thresh = tf.math.reduce_mean(self.corr)
        self.mask = self.corr > self.thresh
        self.mask = tf.cast(self.mask, self.corr.dtype)

        self.masked_corr = tf.math.multiply(self.mask, self.corr)

        self.attention_weights = tf.repeat(self.masked_corr, repeats = self.channels, axis = -1)
        return multiply([self.inputs, self.attention_weights])

    def get_config(self, **kwargs):
        config = super(Pearson_correlation_masked, self).get_config()
        config.update({
            "P_S": self.P_S,
        })
        return config

In [ ]:
@keras_export('keras.layers.Dropout')
class Dropout_Train(layers.Layer):
    def __init__(self, rate, shift = 1, noise_shape=None, seed=None, **kwargs):
        super(Dropout_Train, self).__init__(**kwargs)

        if isinstance(rate, (int, float)) and not 0 <= rate <= 1:
            raise ValueError(f"Invalid value {rate} received for `rate`, expected a value between 0 and 1.")
        if type(shift) != int:
            raise TypeError(f"Invalid dtype {type(shift)} found for `shift`. It must be an integer")
        if shift*rate > 1.0:
            raise ValueError(f"Invalid value {shift} received for `shift`, expected an integer value less than or equal to {int(1/rate)}")
        self.rate = rate
        self.shift = shift
        self.noise_shape = noise_shape
        self.seed = seed
        self.supports_masking = True

    def _get_noise_shape(self, inputs):
        if self.noise_shape is None:
            return None

        concrete_inputs_shape = array_ops.shape(inputs)
        noise_shape = []
        for i, value in enumerate(self.noise_shape):
            noise_shape.append(concrete_inputs_shape[i] if value is None else value)
        return tf.convert_to_tensor(noise_shape)

    def call(self, inputs, training=None):
        if self.rate == 0:
            return tf.identity(inputs)

        if training is None:
            training = K.learning_phase()

        def dropped_inputs():
            input_shape = inputs.shape
            range_0 = int(self.rate*(self.shift-1)*input_shape[-1])
            if self.shift*self.rate < 1.0:
                range_1 = int(self.rate*(self.shift)*input_shape[-1])
            else:
                range_1 = None
            input_shape = inputs.shape
            multiplier = np.ones(input_shape[-1])
            multiplier[range_0:range_1] = 0.0
            multiplier = tf.constant(multiplier)
            return Multiply()([inputs, multiplier])

        output = control_flow_util.smart_cond(training, dropped_inputs, lambda: array_ops.identity(inputs))
        return output

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        config = super(Dropout_Train, self).get_config()
        config.update({
            "rate": self.rate,
            "shift": self.shift,
            "noise_shape": self.noise_shape,
            "seed": self.seed,
            "supports_masking": self.supports_masking
        })
        return config

In [ ]:
def modified_model(model, layer_name, rate, new_layer, shift, **kwargs):    # layer_name = "TRAIN_DROPOUT"
    name = kwargs["name"] if kwargs else None
    x = model.layers[0].output
    modification = False
    z = 0
    for lyr in model.layers[1:]:
        if (layer_name in lyr.name or layer_name in lyr.name.upper()) and (type(shift) != str):
            x = new_layer(rate = rate, shift = shift, name = layer_name + "_" + str(shift) + "_" + str(z))(x)
            modification = True
            z += 1
        elif (layer_name in lyr.name or layer_name in lyr.name.upper()) and (type(shift) == str):
            x = new_layer(rate = rate, name = layer_name + "_" + str(shift) + "_" + str(z))(x)
            z += 1
            modification = True
        else:
            x = lyr(x)
    if not modification:
        print("___________________________________Model has not been modified___________________________________")
    return Model(inputs = model.layers[0].input, outputs = x, name = name)

In [ ]:
class Custom_callbacks(tf.keras.callbacks.Callback):
    def __init__(self, filepath, epochs, rate, new_layer = Dropout_Train, layer_name = "DROPOUT", accuracy_score = 0.99, min_epochs = 50):
        super(Custom_callbacks, self).__init__()
        self.filepath = filepath
        self.epochs = epochs
        self.new_layer = new_layer
        self.rate = rate
        self.best = 0.0
        self.epoch_num = 1
        self.layer_name = layer_name
        self.min_epochs = min_epochs                    # minimum number of epochs that model should be trained in each shift
        self.accuracy_score = accuracy_score if accuracy_score <= 1.0 else accuracy_score/100.0

    def on_train_begin(self, logs=None):
        print(self.epochs)
        keys = list(logs.keys())
        self.shift = 1
        self.epoch_completed = 0
        print(f"Model will be trained in {int(1/self.rate)} shifts")
        print("Starting training with 1st shift \n")
        self.model = modified_model(self.model, self.layer_name, self.rate, self.new_layer, self.shift)

    def on_train_end(self, logs=None):
        keys = list(logs.keys())
        if self.shift <= int(1/self.rate):
            raise NotImplementedError(f"model has not trained fully in the available no. of epochs \n only {self.shift-1} shifts completed out of {int(1/self.rate)}")
        print("Model training completition ", "███████████"*self.shift, (self.rate*(self.shift-1))*100, "%")
        print(f"Model has been fully trained in {int(1/self.rate)} shifts")
        self.model.set_weights(self.best_weights)
        print(f"\nSaving best model to {self.filepath}")
        self.model.save(self.filepath)

    def on_epoch_end(self, epoch, logs=None):
        keys = list(logs.keys())
        self.epoch_completed += 1
        self.epoch_num += 1

        if (logs["val_accuracy"] >= self.accuracy_score) and (self.epoch_completed >= self.min_epochs) and (self.shift < int(1/self.rate)):
            print("\nTargeted accuracy has been achieved")
            print("Model training completition ", "███████████"*(self.shift), (self.rate*self.shift)*100, "%")
            self.shift += 1
            Suffixes = "nd" if (self.shift == 2) else "th"
            print(f"Modifying the model for {self.shift}{str(Suffixes)} shift")
            self.model = modified_model(self.model, self.layer_name, self.rate, self.new_layer, self.shift)
            self.epoch_completed = 0

        elif (logs["val_accuracy"] >= self.accuracy_score) and (self.epoch_completed >= self.min_epochs) and (self.shift == int(1/self.rate)):
            print("\nModel training completition ", "███████████"*(self.shift), (self.rate*self.shift)*100, "%")
            print("All shifting has been completed\n")
            print("██████████████████████===============> Now redefining the model to standard model <===============██████████████████████")
            self.model = modified_model(self.model, self.layer_name, self.rate, self.new_layer, "Final", name = "AlexNet")
            self.shift += 1
            self.epoch_completed = 0

        else:
            print(", need more training")
            if self.shift >= int(1/self.rate):
                current = logs.get("val_accuracy")
                if not np.less(current, self.best) and (self.epoch_num >= self.epochs-10):
                    print(f"val_accuracy improved from {self.best:.4f} to {current:.4f}")
                    self.best = current
                    self.best_weights = self.model.get_weights()

    def get_config(self):
        config = super(Custom_callbacks, self).get_config()
        config.update({
            "filepath": self.filepath,
            "epochs": self.epochs,
            "new_layer": self.new_layer,
            "rate": self.rate,
            "best": self.best,
            "epoch_num": self.epoch_num,
            "layer_name": self.layer_name,
            "min_epochs": self.min_epochs,
            "accuracy_score": self.accuracy_score,
        })
        return config

In [ ]:
def AlexNet(input_shape, num_classes = 13):
    x_input = Input(input_shape)
    if use_pearson_corr:
        X = Pearson_correlation_masked(P_S)(x_input)
    else:
        X = x_input

    X = Conv2D(filters=96, kernel_size =(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=256, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=384, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=384, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = Conv2D(filters=256, kernel_size=(3,3), activation='relu', strides=(1,1), padding='same')(X)
    X = MaxPooling2D(pool_size=(2,2), strides=(2,2), padding='same')(X)

    X = Flatten()(X)
    X = Dense(4096, activation='relu')(X)
    X = Dropout(dropout_rate, name = "TRAIN_DROPOUT_1")(X)
    X = Dense(1024, activation='relu')(X)
    X = Dropout(dropout_rate, name = "TRAIN_DROPOUT_2")(X)
    X = Dense(256, activation='relu')(X)
    X = Dropout(dropout_rate, name = "TRAIN_DROPOUT_3")(X)
    X = Dense(32, activation='relu')(X)

    output = Dense(num_classes, activation='softmax')(X)

    return Model(inputs = x_input, outputs = output, name = "AlexNet")

model = AlexNet(input_shape = (P_S, P_S, B), num_classes = num_classes)
model.summary()

In [ ]:
# plot model architecture
#plot_model(model,to_file = folder_path + "model_architecture.png", show_shapes=False)

# **Training**

In [ ]:
LR_START = 0.01
LR_MAX = 0.02
LR_MIN = 0.005
LR_RAMPUP_EPOCHS = 0
LR_SUSTAIN_EPOCHS = 0
EPOCHS = epoch
STEPS = [epoch,epoch*2]

def lrfn(epoch):
    if epoch<STEPS[0]:
        epoch2 = epoch
        EPOCHS2 = STEPS[0]
    elif epoch<STEPS[0]+STEPS[1]:
        epoch2 = epoch-STEPS[0]
        EPOCHS2 = STEPS[1]
    elif epoch<STEPS[0]+STEPS[1]+STEPS[2]:
        epoch2 = epoch-STEPS[0]-STEPS[1]
        EPOCHS2 = STEPS[2]

    if epoch2 < LR_RAMPUP_EPOCHS:
        lr = (LR_MAX - LR_START) / LR_RAMPUP_EPOCHS * epoch2 + LR_START
    elif epoch2 < LR_RAMPUP_EPOCHS + LR_SUSTAIN_EPOCHS:
        lr = LR_MAX
    else:
        decay_total_epochs = EPOCHS2 - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS - 1
        decay_epoch_index = epoch2 - LR_RAMPUP_EPOCHS - LR_SUSTAIN_EPOCHS
        phase = math.pi * decay_epoch_index / decay_total_epochs
        cosine_decay = 0.5 * (1 + math.cos(phase))
        lr = (LR_MAX - LR_MIN) * cosine_decay + LR_MIN
    return lr

rng = [i for i in range(EPOCHS)]
lr_y = [lrfn(x) for x in rng]
plt.figure(figsize=(10, 4))
plt.plot(rng, lr_y, '-o')
print("Learning rate schedule: {:.3g} to {:.3g} to {:.3g}". \
          format(lr_y[0], max(lr_y), lr_y[-1]))
lr_callback = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose = True)
plt.xlabel('Epoch',size=14)
plt.ylabel('Learning Rate',size=14)
plt.show()

In [ ]:
optimizer = optimizers.SGD(learning_rate=LR_START, momentum=0.9, nesterov=False)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

filepath = folder_path + "Trained models/AlexNet.h5"

callbacks_list = [Custom_callbacks(filepath = filepath, epochs = epoch, rate = dropout_rate, accuracy_score = Targeted_accuracy, min_epochs = Min_trainable_epoch), lr_callback]
# callbacks_list = [ModelCheckpoint(filepath, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max'), lr_callback]

start = time.time()
history = model.fit(x_train, y_train, validation_data=(x_test, y_test), batch_size=batch_size, epochs=epoch, callbacks=callbacks_list, verbose=1)
end = time.time()
tt = end - start
print("Total training time: ", tt)

# **Model Performance**

In [ ]:
plot_accuracy_loss_curve(history, use_pearson_corr = use_pearson_corr, folder_path = folder_path)

In [ ]:
model.load_weights(filepath)
model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

y_pred = predict(model, x_test)

Trainable_params = 0
for i in model.trainable_weights :
    Trainable_params+= np.prod(i.shape)

Non_trainable_params = 0
for i in model.non_trainable_weights:
    Non_trainable_params+= np.prod(i.shape)

Total_params = Trainable_params + Non_trainable_params
parameters_summary = [Total_params, Trainable_params, Non_trainable_params]

performance_meausures(y_test, y_pred, tt, *parameters_summary, folder_path = folder_path)

# **Spatial Adaptive Conformal Prediction (SACP)**

In [ ]:
class SpatialConformalPredictor:
    def __init__(self, height, width, num_classes, lambda_=0.5, alpha=0.05, k=1):
        self.H = height
        self.W = width
        self.num_classes = num_classes
        self.lmd = lambda_ # Smoothing weight
        self.alpha = alpha # Error rate (e.g., 0.05 for 95% coverage)
        self.k = k # Number of smoothing iterations

    def compute_aps_scores(self, probabilities, labels=None):
        """
        Adaptive Prediction Sets (APS) Score Function.
        Returns non-conformity scores.
        """
        n = probabilities.shape[0]
        # Sort probabilities in descending order
        sorted_indices = np.argsort(probabilities, axis=1)[:, ::-1]
        sorted_probs = np.take_along_axis(probabilities, sorted_indices, axis=1)
        
        # Compute cumulative sum
        cumsum = np.cumsum(sorted_probs, axis=1)
        
        # Randomized APS score
        rng = np.random.default_rng(42)
        U = rng.random(n)
        
        if labels is not None:
            # Calculation for Calibration set (Ground Truth known)
            scores = np.zeros(n)
            for i in range(n):
                true_label = labels[i]
                # Find rank of true label
                rank = np.where(sorted_indices[i] == true_label)[0][0]
                
                if rank == 0:
                    scores[i] = U[i] * sorted_probs[i, 0]
                else:
                    scores[i] = cumsum[i, rank-1] + U[i] * sorted_probs[i, rank]
            return scores
        else:
            # Calculation for Test set (All possible labels)
            # Returns matrix of scores for every class
            scores_matrix = np.zeros_like(probabilities)
            for i in range(n):
                # Score if class j were the true class
                # L_j = cumsum_{pi(y) < pi(j)} pi(y) + U * pi(j)
                # We can vectorized computation using the sorted arrays
                scores_sorted = np.zeros(self.num_classes)
                scores_sorted[0] = U[i] * sorted_probs[i, 0]
                scores_sorted[1:] = cumsum[i, :-1] + U[i] * sorted_probs[i, 1:]
                
                # Map back to original indices
                scores_matrix[i] = np.zeros(self.num_classes)
                np.put(scores_matrix[i], sorted_indices[i], scores_sorted)
            return scores_matrix

    def spatial_smoothing(self, score_map, mask_map):
        """
        Applies spatial smoothing to the score map.
        score_map: (H, W, C)
        mask_map: (H, W) boolean, True if pixel contains valid data
        """
        smoothed_map = np.copy(score_map)
        H, W, C = score_map.shape
        
        # Offsets for 8 neighbors
        neighbors = [(-1, -1), (-1, 0), (-1, 1), 
                     (0, -1),           (0, 1), 
                     (1, -1),  (1, 0),  (1, 1)]
        
        # We iterate only over valid pixels to save time
        # In production, convolution is faster, but this matches sacp.py logic
        valid_rows, valid_cols = np.where(mask_map)
        
        for r, c in zip(valid_rows, valid_cols):
            ori_score = score_map[r, c]
            neighbor_sum = np.zeros(C)
            count = 0
            
            for dr, dc in neighbors:
                nr, nc = r + dr, c + dc
                if 0 <= nr < H and 0 <= nc < W:
                    if mask_map[nr, nc]: # Check if neighbor has data
                        neighbor_sum += score_map[nr, nc]
                        count += 1
            
            if count > 0:
                # Formula: lambda * original + lambda * (average of neighbors)
                # Note: sacp.py uses lmd for both. Ensure lmd <= 0.5 usually.
                smoothed_map[r, c] = self.lmd * ori_score + self.lmd * (neighbor_sum / count)
                
        return smoothed_map

    def fit_calibrate(self, calib_probs, calib_labels, calib_indices, 
                      test_probs, test_indices):
        """
        Runs the full SACP pipeline.
        """
        print("1. Computing Base Scores (APS)...")
        # 1. Compute Base Scores
        # calib_scores = self.compute_aps_scores(calib_probs, calib_labels) # We need full matrix for smoothing
        test_scores_matrix = self.compute_aps_scores(test_probs) # (N_test, C)
        calib_scores_matrix = self.compute_aps_scores(calib_probs) # (N_calib, C)
        
        # 2. Map scores to Spatial Grid (H, W, C)
        print("2. Mapping to Spatial Grid...")
        score_map = np.zeros((self.H, self.W, self.num_classes))
        mask_map = np.zeros((self.H, self.W), dtype=bool)
        
        for idx, (r, c) in enumerate(calib_indices):
            score_map[r, c] = calib_scores_matrix[idx]
            mask_map[r, c] = True
            
        for idx, (r, c) in enumerate(test_indices):
            score_map[r, c] = test_scores_matrix[idx]
            mask_map[r, c] = True
            
        # 3. Spatial Fusion (Smoothing)
        print(f"3. Running Spatial Fusion (k={self.k})...")
        current_map = score_map
        for _ in tqdm(range(self.k), desc="Smoothing"):
            current_map = self.spatial_smoothing(current_map, mask_map)
            
        # 4. Extract Fused Scores for Calibration to find Q_hat
        fused_calib_scores = []
        for idx, (r, c) in enumerate(calib_indices):
            # For q_hat, we need the score of the TRUE label
            true_lbl = calib_labels[idx]
            fused_calib_scores.append(current_map[r, c, true_lbl])
            
        fused_calib_scores = np.array(fused_calib_scores)
        
        # 5. Compute Q_hat (Conformal Quantile)
        n = len(fused_calib_scores)
        q_level = np.ceil((n + 1) * (1 - self.alpha)) / n
        q_level = min(1.0, max(0.0, q_level)) # Clip
        q_hat = np.quantile(fused_calib_scores, q_level, method='higher')
        
        print(f"   Q_hat determined: {q_hat:.4f}")
        
        # 6. Generate Prediction Sets for Test Data
        prediction_sets = []
        avg_size = 0
        
        for idx, (r, c) in enumerate(test_indices):
            # Get fused scores for this test point
            scores = current_map[r, c]
            # Construct set: Class is included if its score <= q_hat
            pset = np.where(scores <= q_hat)[0]
            prediction_sets.append(pset)
            avg_size += len(pset)
            
        avg_size /= len(prediction_sets)
        
        return prediction_sets, q_hat, avg_size

In [ ]:
# --- EXECUTE SACP ---

# 1. Get Probabilities from your trained Keras model
print("Predicting probabilities for Calibration and Test sets...")
# Ensure 'model' is your trained Keras model
probs_calib = model.predict(x_calib, verbose=1)
probs_test = model.predict(x_test, verbose=1)

# 2. Initialize SACP
# H, W from your existing variables
# num_classes is defined in your Data section
sacp = SpatialConformalPredictor(
    height=H, 
    width=W, 
    num_classes=num_classes, 
    lambda_=SACP_LAMBDA, 
    alpha=SACP_ALPHA,   
    k=SACP_K           
)

# 3. Run Calibration and Prediction
pred_sets, q_hat, avg_set_size = sacp.fit_calibrate(
    probs_calib, y_calib, idx_calib,
    probs_test, idx_test
)

# 4. Evaluate Coverage (Accuracy of the sets)
coverage_count = 0
for i, pset in enumerate(pred_sets):
    if y_test[i] in pset:
        coverage_count += 1

coverage = coverage_count / len(y_test)

print("-" * 30)
print("SACP RESULTS")
print("-" * 30)
print(f"Target Coverage (1-alpha): {1 - sacp.alpha:.2f}")
print(f"Actual Coverage:           {coverage:.4f}")
print(f"Average Set Size:          {avg_set_size:.2f}")
print("-" * 30)

# Optional: Visualize a Prediction Set Size Map
# This creates a map showing how 'uncertain' different areas are
size_map = np.zeros((H, W))
for i, (r, c) in enumerate(idx_test):
    size_map[r, c] = len(pred_sets[i])

plt.figure(figsize=(10, 10))
plt.imshow(size_map, cmap='viridis')
plt.colorbar(label='Prediction Set Size')
plt.title('Uncertainty Map (Set Size)')
plt.show()